# 🧠 MindSet Pro — Complete AI-Powered Habit & Task Intelligence System\n### *Fully Integrated, Production-Grade Notebook*

## ⚙️ Setup and Library Installation

In [ ]:
!apt-get update && apt-get install -y portaudio19-dev python3-dev\n!pip install SpeechRecognition\n!pip install firebase-admin\n!pip install pyaudio\

## 📦 Imports and Global Configuration

In [ ]:
import os, re, sys, json, uuid, math, random, hashlib, warnings, textwrap
import time
from datetime import datetime, date, timedelta, timezone
from typing import Optional, Dict, List, Tuple, Literal, Any
from dataclasses import dataclass, field, asdict
from collections import defaultdict, Counter
from enum import Enum

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
import speech_recognition as sr
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, silhouette_score
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

FIREBASE_ENABLED = False
TODAY = date.today()

PALETTE = {
    'accent':   '#7fffb2', 'accent2':  '#5b8fff', 'accent3':  '#ff6b9d', 'accent4':  '#ffd166',
    'bg':       '#060608', 'surface':  '#0e0e14', 'surface2': '#16161f', 'surface3': '#1e1e2a',
    'text':     '#f0f0f8', 'text2':    '#8888a8', 'text3':    '#55556a',
}

CATEGORIES  = ['Health', 'Work', 'Learning', 'Mindfulness', 'Personal', 'Finance']
PRIORITIES  = ['High', 'Medium', 'Low']
DAYS_LABELS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

print(f"✅ Environment ready  |  Python {sys.version.split()[0]}  |  Today: {TODAY}")\

## 📐 Phase 1 — Data Models & Schema Definitions

In [ ]:
class Priority(str, Enum): HIGH='High'; MEDIUM='Medium'; LOW='Low'
class Category(str, Enum): HEALTH='Health'; WORK='Work'; LEARNING='Learning'; MINDFULNESS='Mindfulness'; PERSONAL='Personal'; FINANCE='Finance'
class Intent(str, Enum): ADD_TASK='add_task'; COMPLETE_TASK='complete_task'; DELETE_TASK='delete_task'; ADD_HABIT='add_habit'; TOGGLE_HABIT='toggle_habit'; QUERY_STREAK='query_streak'; QUERY_TASKS='query_tasks'; LIST_HABITS='list_habits'; UNKNOWN='unknown'

@dataclass
class Task:
    name: str
    category: str = 'Work'
    priority: str = 'Medium'
    done: bool = False
    date: Optional[str] = None
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    due_time: Optional[datetime] = None
    completed: bool = False
    
    def __post_init__(self): self.completed = self.done
    def to_dict(self) -> dict:
        d = asdict(self)
        if self.due_time and isinstance(self.due_time, datetime): d['due_time'] = self.due_time.isoformat()
        return d

@dataclass
class Habit:
    name: str
    emoji: str = '🎯'
    category: str = 'Health'
    target_days: List[int] = field(default_factory=lambda: [1,2,3,4,5,6,7])
    history: Dict[str, bool] = field(default_factory=dict)
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def to_dict(self) -> dict: return asdict(self)
    @property
    def total_completions(self) -> int: return sum(1 for v in self.history.values() if v)
    @property
    def streak(self) -> int:
        s = 0; d = TODAY - timedelta(days=1)
        while True:
            if self.history.get(d.isoformat()): s += 1; d -= timedelta(days=1)
            else: break
            if s > 365: break
        if self.history.get(TODAY.isoformat()): s += 1
        return s
    @property
    def longest_streak(self) -> int:
        if not self.history: return 0
        dates = sorted(k for k, v in self.history.items() if v)
        if not dates: return 0
        best = cur = 1
        for i in range(1, len(dates)):
            if (date.fromisoformat(dates[i]) - date.fromisoformat(dates[i-1])).days == 1: cur += 1; best = max(best, cur)
            else: cur = 1
        return best
    def completion_rate(self, days: int = 30) -> float:
        done = possible = 0; d = TODAY
        for _ in range(days):
            if d.isoweekday() in self.target_days:
                possible += 1
                if self.history.get(d.isoformat()): done += 1
            d -= timedelta(days=1)
        return round(done / possible, 4) if possible else 0.0

@dataclass
class VoiceResult:
    success: bool
    intent: str
    confidence: float
    message: str
    data: Optional[dict] = None
    raw_command: str = ''
\

## 🏭 Phase 2 — Synthetic Data Engine

In [ ]:
class SyntheticDataEngine:
    def __init__(self, seed: int = 42): random.seed(seed); np.random.seed(seed)
    def _seeded_prob(self, habit_id: str, base: float = 0.68) -> float:
        h = int(hashlib.md5(habit_id.encode()).hexdigest(), 16); return max(0.30, min(0.92, base + ((h % 100) - 50) / 500))
    def generate_habit_history(self, habit: Habit, days: int = 180, base_prob: float = 0.68) -> Dict[str, bool]:
        history = {}; prob = self._seeded_prob(habit.id, base_prob); burst_left = cold_left = 0
        for i in range(days - 1, -1, -1):
            d = TODAY - timedelta(days=i)
            if d.isoweekday() not in habit.target_days: continue
            trend_prob = min(0.95, prob + 0.002 * ((days - i) // 7))
            if d.isoweekday() >= 6: trend_prob *= 0.82
            if burst_left > 0: trend_prob = min(0.97, trend_prob * 1.4); burst_left -= 1
            elif cold_left > 0: trend_prob *= 0.35; cold_left -= 1
            elif random.random() < 0.04: burst_left = random.randint(3, 7)
            elif random.random() < 0.03: cold_left = random.randint(2, 4)
            if random.random() < trend_prob: history[d.isoformat()] = True
        return history
    def generate_habits(self) -> List[Habit]:
        spec = [('Morning Run','🏃','Health',[1,2,3,4,5],0.72), ('Read 30 mins','📚','Learning',[1,2,3,4,5,6,7],0.79), ('Drink 8 Glasses','💧','Health',[1,2,3,4,5,6,7],0.85)]
        habits = []
        for name, emoji, cat, days, prob in spec:
            h = Habit(name=name, emoji=emoji, category=cat, target_days=days)
            h.history = self.generate_habit_history(h, 180, prob)
            habits.append(h)
        return habits
    def generate_tasks(self, count: int = 15) -> List[Task]:
        tasks = []
        for _ in range(count):
            t = Task(name=f"Task {random.randint(1,100)}", category="Work", priority="High" if random.random() < 0.3 else "Medium")
            t.done = random.random() < 0.5
            tasks.append(t)
        return tasks

engine = SyntheticDataEngine(seed=42)
HABITS = engine.generate_habits()
TASKS  = engine.generate_tasks(count=20)
\

## 📋 Phase 3 — Task Manager (Full CRUD Engine)

In [ ]:
class TaskManager:
    def __init__(self, initial_tasks=None):
        self._store = {}
        if initial_tasks:
            for t in initial_tasks: self._store[t.id] = t
    def create(self, name, category='Work', priority='Medium', date_str=None):
        t = Task(name=name, category=category, priority=priority, date=date_str or TODAY.isoformat())
        self._store[t.id] = t; return t
    def get(self, task_id): return self._store.get(task_id)
    def update(self, task_id, **kwargs):
        t = self._store.get(task_id)
        if t:
            for k, v in kwargs.items():
                if hasattr(t, k): setattr(t, k, v)
        return t
    def delete(self, task_id): return bool(self._store.pop(task_id, None))
    def mark_done(self, task_id, done=True): return self.update(task_id, done=done)
    def all(self, category=None, priority=None, done=None, sort_by='priority'):
        tasks = list(self._store.values())
        if done is not None: tasks = [t for t in tasks if t.done == done]
        return tasks
    def find_by_name(self, fragment):
        for t in self._store.values():
            if fragment.lower() in t.name.lower(): return t
        return None

TM = TaskManager(initial_tasks=TASKS)
\

## 🔄 Phase 4 — Habit Manager (Full CRUD + Streak Engine)

In [ ]:
class HabitManager:
    def __init__(self, initial_habits=None):
        self._store = {}
        if initial_habits:
            for h in initial_habits: self._store[h.id] = h
    def all(self, category=None): return list(self._store.values())

HM = HabitManager(initial_habits=HABITS)
\

## 📝 Phase 11 — Interactive Todo List

This cell directly queries the `TaskManager` and displays our tasks clearly formatted as an interactive Todo List.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

def show_todo_list(task_mgr):
    tasks = task_mgr.all()
    # Sort priority: High -> Medium -> Low
    pri_map = {'High':1, 'Medium':2, 'Low':3}
    tasks.sort(key=lambda t: (t.done, pri_map.get(t.priority, 4)))
    
    html = "<div style='background-color:#0e0e14; padding:20px; border-radius:10px; color:#f0f0f8; font-family:monospace;'>"
    html += "<h2 style='color:#7fffb2; margin-top:0;'>📋 MindSet Pro - Todo List Dashboard</h2>"
    html += "<table style='width: 100%; border-collapse: collapse; text-align: left;'>"
    html += "<tr style='border-bottom: 2px solid #2a2a3a; color:#8888a8;'>"
    html += "<th style='padding:10px;'>Status</th><th style='padding:10px;'>Task</th>"
    html += "<th style='padding:10px;'>Category</th><th style='padding:10px;'>Priority</th></tr>"
    
    for t in tasks:
        icon = "✅" if t.done else "🔲"
        color = "#ff6b9d" if t.priority=='High' and not t.done else ("#5b8fff" if t.priority=='Medium' and not t.done else ("#7fffb2" if t.done else "#8888a8"))
        strike = "text-decoration: line-through; opacity: 0.6;" if t.done else ""
        html += f"<tr style='border-bottom: 1px solid #1e1e2a; {strike}'>"
        html += f"<td style='padding:10px; font-size:16px;'>{icon}</td>"
        html += f"<td style='padding:10px; color:{color}; font-weight:bold;'>{t.name}</td>"
        html += f"<td style='padding:10px; color:#a0a0c0;'>{t.category}</td>"
        html += f"<td style='padding:10px; color:{color};'>{t.priority}</td>"
        html += "</tr>"
        
    html += "</table></div>"
    display(HTML(html))

show_todo_list(TM)
\